# Olist Brazilian E-Commerce: Why are customers leaving bad reviews?
### SQL analysis with DuckDB and Jupyter

This notebook investigates whether late deliveries explain falling customer satisfaction at a Brazilian marketplace, using about 99k orders from the public Olist dataset. It answers eight business questions in SQL: delivery delays, review scores, product categories, sellers, regional delivery times, revenue by state, order size and missing reviews.

In [ ]:
from pathlib import Path
import duckdb

BASE_DIR = Path().resolve().parent
db_path = BASE_DIR / "database" / "olist.duckdb"

# Connects via DuckDB
con = duckdb.connect(str(db_path), read_only=True)

# Passes the existing connection to the SQL extension
%reload_ext sql
%sql con
%config SqlMagic.autolimit = 50

# To display, for example, up to 50 rows in the table:
%config SqlMagic.displaylimit = 15

## Olist Brazilian E-Commerce: "Why are customers leaving bad reviews?"

Our teams noticed that customer satisfaction has been decreasing and our team thinks it was caused due to the late deliveries. to answer that we need to do a set of queries to find out.

### What percentage of orders are delivered later than their estimated delivery date?

In [4]:
%%sql
--to answer this question we´ll be using the CASE.
--the CASE function allows us to filter when a delivery was a delay or not using date arithmetic
WITH orders_relation AS (
    SELECT
        CASE
            WHEN (order_delivered_customer_date > order_estimated_delivery_date) THEN 'delayed'
            ELSE 'on_time'
        END AS delivery_status
    FROM orders
    WHERE order_status = 'delivered' 
      AND order_delivered_customer_date IS NOT NULL 
      AND order_estimated_delivery_date IS NOT NULL
)

/************************************************************************************************
after that we need to sum all the delayed orders.
we need to sum where the status_entrea = delayed/not_delayed, if it´s THEN 1 if not THEN 0
but when we devide an integer by another integer sql omites the decimal value letting a inprecise
number, so to solve that we use the cast operator ::decimal it converts the number on its left
turning it into a decimal value.
*************************************************************************************************/

SELECT
    (SUM(CASE WHEN delivery_status = 'delayed' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pkg_delayed,
    (SUM(CASE WHEN delivery_status = 'on_time' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pkg_ontime
FROM orders_relation;


Running query in 'DuckDBPyConnection'

pkg_delayed,pkg_ontime
8.112366538820359,91.88763346117965


> **Conclusion:** 8.11% of delivered orders arrived after the estimated date. Late orders are a minority, so other factors may also be influencing customer dissatisfaction.

## Is there a relationship between delivery delay and review score? (e.g., average review score for on-time vs. late orders)

In [5]:
%%sql
WITH delivery AS (
    SELECT
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'late'
            ELSE 'on_time'
        END AS delivery_status,
        r.review_score
    FROM orders o
    JOIN order_reviews r
        ON r.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)
SELECT
    delivery_status,
    COUNT(*) AS reviews,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM delivery
GROUP BY delivery_status;


Running query in 'DuckDBPyConnection'

delivery_status,reviews,avg_review_score
late,7700,2.57
on_time,88653,4.29


In [6]:
%%sql
WITH delivery AS (
    SELECT
        CEIL(DATE_DIFF('second', o.order_estimated_delivery_date, o.order_delivered_customer_date) / 86400.0) AS days_late,
        r.review_score
    FROM orders o
    JOIN order_reviews r
        ON r.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)
SELECT
    CASE
        WHEN days_late <= 0 THEN '0. on time'
        WHEN days_late <= 3 THEN '1. 1-3 days late'
        WHEN days_late <= 7 THEN '2. 4-7 days late'
        ELSE '3. more than 7 days late'
    END AS delay_bucket,
    COUNT(*) AS reviews,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM delivery
GROUP BY delay_bucket
ORDER BY delay_bucket;

Running query in 'DuckDBPyConnection'

delay_bucket,reviews,avg_review_score
0. on time,88653,4.29
1. 1-3 days late,2651,3.77
2. 4-7 days late,1777,2.32
3. more than 7 days late,3272,1.73


> **Conclusion:** Late orders average 2.57 stars versus 4.29 for on-time orders, and the score keeps falling as the delay grows: 3.77 for 1-3 days late, 2.32 for 4-7 days and 1.73 for more than 7 days. Delay is strongly associated with low ratings, although this analysis does not prove causation.

## Which product categories have the worst average review scores? Which have the best?

In [7]:
%%sql
WITH order_category AS (
    -- one row per order x category, so an order with 3 items of the same category counts once
    SELECT DISTINCT
        oi.order_id,
        COALESCE(p.product_category_name, 'unknown') AS category
    FROM order_items oi
    JOIN products p
        ON p.product_id = oi.product_id
),
category_scores AS (
    SELECT
        oc.category,
        COUNT(*) AS reviews,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM order_category oc
    JOIN order_reviews r
        ON r.order_id = oc.order_id
    GROUP BY oc.category
    HAVING COUNT(*) >= 100   -- ignore categories with too few reviews to be reliable
)
(SELECT 'best' AS ranking, * FROM category_scores ORDER BY avg_review_score DESC LIMIT 5)
UNION ALL
(SELECT 'worst' AS ranking, * FROM category_scores ORDER BY avg_review_score ASC LIMIT 5);

Running query in 'DuckDBPyConnection'

ranking,category,reviews,avg_review_score
best,livros_interesse_geral,508,4.46
best,livros_tecnicos,259,4.41
best,alimentos_bebidas,228,4.39
best,malas_acessorios,1030,4.33
best,alimentos,445,4.28
worst,moveis_escritorio,1268,3.62
worst,fashion_roupa_masculina,111,3.7
worst,audio,348,3.83
worst,construcao_ferramentas_seguranca,166,3.85
worst,casa_conforto,398,3.86


> **Conclusion:** Among categories with at least 100 reviews, average scores range from 3.62 (office furniture) to 4.46 (general-interest books). Books and food rank best. Office furniture is the weakest and, with 1,268 reviews, the most reliable of the low scores. Category differences (about 0.8 stars) are much smaller than the gap between on-time orders and orders more than 7 days late (about 2.6 stars), so categories explain less of the dissatisfaction than delivery delays do.


## **Which sellers have the highest rate of late deliveries? Should any be flagged for a performance review?**

In [8]:
%%sql
WITH seller_orders AS (
    -- one row per seller x order (an order with 3 items from the same seller counts once)
    SELECT
        oi.seller_id,
        o.order_id,
        MAX(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS is_late
    FROM orders o
    JOIN order_items oi
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY oi.seller_id, o.order_id
)
SELECT
    seller_id,
    COUNT(*) AS orders,
    SUM(is_late) AS late_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) AS late_rate_pct
FROM seller_orders
GROUP BY seller_id
HAVING COUNT(*) >= 30   -- minimum volume so one unlucky order does not make a 100% late rate
ORDER BY late_rate_pct DESC, orders DESC
LIMIT 15;


Running query in 'DuckDBPyConnection'

seller_id,orders,late_orders,late_rate_pct
ede0c03645598cdfc63ca8237acbe73d,43,15,34.88
ad781527c93d00d89a11eecd9dcad7c1,38,12,31.58
835f0f7810c76831d6c7d24c7a646d4d,42,13,30.95
54965bbe3e4f07ae045b90b0b8541f52,73,22,30.14
2a1348e9addc1af5aaa619b1a3679d6b,48,13,27.08
66e0557ecc2b4dbea057e93f215f68d8,30,8,26.67
a49928bcdf77c55c6d6e05e09a9b4ca5,96,25,26.04
ef990a83bbea832f36ebe81376335aa8,43,11,25.58
beadbee30901a7f61d031b6b686095ad,64,16,25.0
06a2c3af7b3aee5d69171b0e14f0ee87,389,90,23.14


In [9]:
%%sql
SELECT
    ROUND(AVG((order_delivered_customer_date > order_estimated_delivery_date)::INT) * 100, 2) AS platform_late_rate_pct
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;

Running query in 'DuckDBPyConnection'

platform_late_rate_pct
8.11


In [16]:
%%sql
WITH seller_orders AS (
    SELECT
        oi.seller_id,
        o.order_id,
        MAX(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS is_late
    FROM orders o
    JOIN order_items oi
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY oi.seller_id, o.order_id
),
seller_rates AS (
    SELECT
        seller_id,
        COUNT(*) AS orders,
        SUM(is_late) AS late_orders,
        SUM(is_late) * 100.0 / COUNT(*) AS late_rate_pct
    FROM seller_orders
    GROUP BY seller_id
    HAVING COUNT(*) >= 30
)
SELECT
    COUNT(*) AS sellers_with_30_plus_orders,
    COUNT(*) FILTER (WHERE late_rate_pct > 2 * 8.11) AS flagged_sellers,
    SUM(late_orders) FILTER (WHERE late_rate_pct > 2 * 8.11) AS late_orders_from_flagged,
    SUM(late_orders) AS late_orders_all_30_plus
FROM seller_rates;

Running query in 'DuckDBPyConnection'

sellers_with_30_plus_orders,flagged_sellers,late_orders_from_flagged,late_orders_all_30_plus
627,40,678,6523


> **Conclusion:** Using a minimum of 30 orders per seller to avoid unreliable rates, **40** sellers have a late rate above 16.22% (twice the 8.11% platform average), and together they account for **678** late orders. Among the 15 highest rates, the worst reaches 34.9%. Seller 06a2c3af... stands out for impact: 389 orders with 23.1% late, which is 90 late orders. These sellers should be reviewed first. Limitation: when an order has items from two sellers, the delay is attributed to both.

## What is the average time between order purchase and order delivery, broken down by state/region?

In [10]:
%%sql
SELECT
    c.customer_state AS state,
    COUNT(*) AS quantity,
    ROUND(AVG(DATE_DIFF('seconds', order_purchase_timestamp, order_delivered_customer_date)) / 86400.0, 1) AS avg_purchase_to_delivery
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
    AND o.order_purchase_timestamp IS NOT NULL
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_purchase_to_delivery DESC;



Running query in 'DuckDBPyConnection'

state,quantity,avg_purchase_to_delivery
RR,41,29.4
AP,67,27.2
AM,145,26.4
AL,397,24.5
PA,946,23.8
MA,717,21.6
SE,335,21.5
CE,1279,21.3
AC,80,21.0
PB,517,20.4


> **Conclusion:** Customers in RR, AP and AM wait roughly 4 weeks on average (29.4, 27.2 and 26.4 days), versus about 9 days in SP (8.8, see question 6). RR and AP have few delivered orders (41 and 67), so their averages are less stable.

## Which state generates the most revenue, and which has the slowest average delivery time — is there a link?

In [18]:
%%sql
WITH state_revenue AS (
    SELECT
        c.customer_state AS state,
        ROUND(SUM(op.payment_value)) AS total_revenue
    FROM customers c
    JOIN orders o
        ON o.customer_id = c.customer_id
    JOIN order_payments op
        ON op.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
),
state_delivery AS (
    SELECT
        c.customer_state AS state,
        -- delivery time = delivered - purchased (NOT estimated - delivered)
        ROUND(AVG(DATE_DIFF('second', o.order_purchase_timestamp, o.order_delivered_customer_date)) / 86400.0, 1) AS avg_days_to_deliver
    FROM customers c
    JOIN orders o
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
),
ranked AS (
    SELECT
        sr.state,
        sr.total_revenue,
        sd.avg_days_to_deliver,
        RANK() OVER (ORDER BY sr.total_revenue DESC) AS revenue_rank,
        RANK() OVER (ORDER BY sd.avg_days_to_deliver DESC) AS slowest_rank
    FROM state_revenue sr
    JOIN state_delivery sd
        ON sd.state = sr.state
)
SELECT * FROM ranked ORDER BY revenue_rank;

Running query in 'DuckDBPyConnection'

state,total_revenue,avg_days_to_deliver,revenue_rank,slowest_rank
SP,5770266.0,8.8,1,27
RJ,2055690.0,15.3,2,21
MG,1819278.0,12.0,3,25
RS,861802.0,15.3,4,21
PR,781920.0,12.0,5,25
SC,595208.0,15.0,6,23
BA,591271.0,19.3,7,13
DF,346146.0,13.0,8,24
GO,334294.0,15.6,9,19
ES,317683.0,15.8,10,18


In [17]:
%%sql
WITH state_revenue AS (
    SELECT
        c.customer_state AS state,
        ROUND(SUM(op.payment_value)) AS total_revenue
    FROM customers c
    JOIN orders o
        ON o.customer_id = c.customer_id
    JOIN order_payments op
        ON op.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
),
state_delivery AS (
    SELECT
        c.customer_state AS state,
        -- delivery time = delivered - purchased (NOT estimated - delivered)
        ROUND(AVG(DATE_DIFF('second', o.order_purchase_timestamp, o.order_delivered_customer_date)) / 86400.0, 1) AS avg_days_to_deliver
    FROM customers c
    JOIN orders o
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
),
ranked AS (
    SELECT
        sr.state,
        sr.total_revenue,
        sd.avg_days_to_deliver,
        RANK() OVER (ORDER BY sr.total_revenue DESC) AS revenue_rank,
        RANK() OVER (ORDER BY sd.avg_days_to_deliver DESC) AS slowest_rank
    FROM state_revenue sr
    JOIN state_delivery sd
        ON sd.state = sr.state
)
SELECT ROUND(CORR(revenue_rank, slowest_rank), 2) AS rank_correlation FROM ranked;

Running query in 'DuckDBPyConnection'

rank_correlation
-0.8


> **Conclusion:** Revenue and delivery speed are linked. The highest-revenue states receive orders fastest (SP 8.8 days, MG and PR 12.0), while the slowest states are small North and Northeast markets (RR, AP, AM: 26-29 days). The rank correlation between revenue and slowness is **-0.8**: higher revenue goes with faster delivery. BA is an exception, 7th in revenue but 19.3 days. A possible explanation is distance from where sellers ship, which this analysis does not test. Revenue here is the sum of payments of delivered orders.

## Do orders with more items (multiple products) have different review scores than single-item orders?

In [12]:
%%sql
WITH order_type AS (
    SELECT
        order_id,
        COUNT(*) AS items,
        COUNT(DISTINCT seller_id) AS sellers,
        CASE WHEN COUNT(*) > 1 THEN 'multiple_items' ELSE 'single_item' END AS order_category
    FROM order_items
    GROUP BY order_id
)
SELECT
    ot.order_category,
    COUNT(*) AS reviews,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    ROUND(AVG(ot.sellers), 2) AS avg_sellers_per_order
FROM order_type ot
JOIN order_reviews r
    ON r.order_id = ot.order_id
GROUP BY ot.order_category;
    

Running query in 'DuckDBPyConnection'

order_category,reviews,avg_review_score,avg_sellers_per_order
multiple_items,9766,3.6,1.14
single_item,88699,4.16,1.0


In [13]:
%%sql
WITH order_type AS (
    SELECT
        order_id,
        CASE WHEN COUNT(*) > 1 THEN 'multiple_items' ELSE 'single_item' END AS order_category
    FROM order_items
    GROUP BY order_id
)
SELECT
    ot.order_category,
    COUNT(*) AS delivered_orders,
    ROUND(AVG((o.order_delivered_customer_date > o.order_estimated_delivery_date)::INT) * 100, 2) AS pct_late
FROM order_type ot
JOIN orders o
    ON o.order_id = ot.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY ot.order_category;

Running query in 'DuckDBPyConnection'

order_category,delivered_orders,pct_late
single_item,86835,8.29
multiple_items,9635,6.53


> **Conclusion:** Orders with more than one item score lower (3.60 vs 4.16), even though they are late less often (6.53% vs 8.29%) and have only 1.14 sellers per order on average. So neither delivery delay nor multiple sellers explains the gap. Possible causes, not tested here: partial or split shipments, missing or wrong items, or higher expectations on larger orders.

## What percentage of orders have a missing or null review — and does that skew any of your averages above?

In [14]:
%%sql
SELECT
    COUNT(*) AS total_orders,
    COUNT(*) FILTER (WHERE r.order_id IS NULL) AS orders_without_review,
    ROUND(COUNT(*) FILTER (WHERE r.order_id IS NULL) * 100.0 / COUNT(*), 2) AS pkg_missing_reviews
FROM orders o
LEFT JOIN (
    SELECT DISTINCT order_id
    FROM order_reviews
    WHERE review_score IS NOT NULL
) r
    ON r.order_id = o.order_id;

Running query in 'DuckDBPyConnection'

total_orders,orders_without_review,pkg_missing_reviews
99441,768,0.77


In [15]:
%%sql
SELECT
    o.order_status,
    COUNT(*) AS orders,
    COUNT(*) FILTER (WHERE r.order_id IS NULL) AS orders_without_review
FROM orders o
LEFT JOIN (
    SELECT DISTINCT order_id
    FROM order_reviews
    WHERE review_score IS NOT NULL
) r
    ON r.order_id = o.order_id
GROUP BY o.order_status
ORDER BY orders_without_review DESC;

Running query in 'DuckDBPyConnection'

order_status,orders,orders_without_review
delivered,96478,646
shipped,1107,75
canceled,625,20
unavailable,609,14
processing,301,6
invoiced,314,5
created,5,2
approved,2,0


> **Conclusion:** Only 0.77% of orders (768 of 99,441) have no review. Among delivered orders, which is what the delay analysis uses, it is 0.67% (646 of 96,478); the missing ones concentrate in orders not yet delivered (shipped: 6.8%). Missing reviews therefore barely affect the averages above. Caveat: reviews are voluntary, so customers with very good or very bad experiences may be over-represented.